# Module 7 – Procurement Cost Prediction
## Business Problem
Procurement costs fluctuate because of supplier performance, transportation costs, inventory levels, purchasing behaviour, and market conditions. Predicting procurement costs before purchase orders are issued allows procurement teams to negotiate better contracts and optimize supplier selection.

## Business Objective
Predict procurement invoice cost using operational features available before purchase execution.

## Dataset Description
- **Target Variable**: `procurement_cost_usd`
- **Features**: Product Category, Order Quantity, Region, Vendor Tier, VRIS Score, Defect Rate, Shipment Mode.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
orders = pd.read_csv('../Data/cleaned/orders_cleaned.csv')
vendors = pd.read_csv('../Data/cleaned/vendors_cleaned.csv')
financials = pd.read_csv('../Data/cleaned/financials_cleaned.csv')

df = orders[['order_id', 'product_category', 'order_quantity', 'region', 'vendor_id', 'fulfillment_channel']].merge(
    financials[['order_id', 'procurement_cost_usd']], on='order_id', how='inner'
)
df = df.merge(vendors[['vendor_id', 'vendor_tier', 'vris_score', 'defect_rate_pct']], on='vendor_id', how='left')
df.dropna(subset=['procurement_cost_usd'], inplace=True)
df.rename(columns={'fulfillment_channel': 'shipment_mode'}, inplace=True)

X = df.drop(columns=['order_id', 'vendor_id', 'procurement_cost_usd'])
y = df['procurement_cost_usd']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Model Development and Evaluation

In [ ]:
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X.select_dtypes(include=['number']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42)
}

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    print(f"{name} -> MAE: {mean_absolute_error(y_test, y_pred):.2f}, RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}, R2: {r2_score(y_test, y_pred):.4f}")
